# 21 · Bootstrap Uncertainty for Cross-Ensemble Transfer

This notebook adds bootstrap uncertainty to the transfer experiments from Notebook 20.

## Core question

The transfer accuracies in Notebook 20 were high and stable.  
Now we ask:

> Are those accuracies statistically stable under resampling?

We focus on two reciprocal tasks:

1. **train GUE vs Poisson → test zeta vs Poisson**
2. **train zeta vs Poisson → test GUE vs Poisson**

and compare them to same-ensemble baselines:

- zeta → zeta
- GUE → GUE

## Output

For each height block and each direction, we compute:
- bootstrap mean accuracy
- bootstrap standard deviation
- bootstrap 95% interval

for both:
- nearest centroid
- k-NN (k=5)

In [ ]:
import numpy as np
import mpmath as mp
import matplotlib.pyplot as plt

mp.mp.dps = 50
rng = np.random.default_rng(9423)

## Base data

In [ ]:
N = 1800
zeros = [mp.zetazero(n) for n in range(1, N + 1)]
t = np.array([float(mp.im(z)) for z in zeros])
zeta_gaps_full = np.diff(t)

poisson_gaps_full = rng.exponential(scale=1.0, size=len(zeta_gaps_full))

def sample_gue_bulk_spacings(matrix_size=150, n_matrices=70, bulk_fraction=0.5, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    all_gaps = []
    for _ in range(n_matrices):
        real = rng.normal(size=(matrix_size, matrix_size))
        imag = rng.normal(size=(matrix_size, matrix_size))
        A = real + 1j * imag
        H = (A + A.conj().T) / 2.0
        eigvals = np.linalg.eigvalsh(H)
        eigvals = np.sort(eigvals)

        n = len(eigvals)
        keep = int(n * bulk_fraction)
        start = (n - keep) // 2
        stop = start + keep
        bulk_vals = eigvals[start:stop]
        bulk_gaps = np.diff(bulk_vals)
        all_gaps.extend(bulk_gaps.tolist())

    return np.array(all_gaps)

gue_gaps_full = sample_gue_bulk_spacings(matrix_size=150, n_matrices=70, bulk_fraction=0.5, rng=rng)

len(zeta_gaps_full), len(poisson_gaps_full), len(gue_gaps_full)

## Minimal feature set

In [ ]:
def extract_windows(gaps, k=5):
    return np.array([gaps[i:i+k] for i in range(len(gaps) - k + 1)])

def normalize_window(window):
    w = np.array(window, dtype=float)
    return w / w.mean()

def unevenness(window):
    return float(np.max(window) - np.min(window))

def window_entropy(window):
    eps = 1e-12
    p = window / np.sum(window)
    return float(-np.sum(p * np.log(p + eps)))

def ratio_mean(window):
    g1 = window[:-1]
    g2 = window[1:]
    r = np.minimum(g1, g2) / np.maximum(g1, g2)
    return float(np.mean(r))

def build_feature_matrix(gaps, k=5):
    windows = extract_windows(gaps, k=k)
    windows_n = np.array([normalize_window(w) for w in windows])
    X = np.array([
        [window_entropy(w), unevenness(w), ratio_mean(w)]
        for w in windows_n
    ], dtype=float)
    return windows_n, X

## Classifier helpers

In [ ]:
def standardize_train_test(X_train, X_test):
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0)
    std[std == 0] = 1.0
    return (X_train - mean) / std, (X_test - mean) / std

def nearest_centroid_fit(X_train, y_train):
    classes = np.unique(y_train)
    centroids = {c: X_train[y_train == c].mean(axis=0) for c in classes}
    return centroids

def nearest_centroid_predict(X, centroids):
    classes = sorted(centroids.keys())
    centroid_array = np.vstack([centroids[c] for c in classes])
    dists = np.linalg.norm(X[:, None, :] - centroid_array[None, :, :], axis=2)
    pred_idx = np.argmin(dists, axis=1)
    return np.array([classes[i] for i in pred_idx])

def knn_predict(X_train, y_train, X_test, k=5):
    preds = []
    for x in X_test:
        d = np.linalg.norm(X_train - x, axis=1)
        nn = np.argsort(d)[:k]
        votes = y_train[nn]
        counts = np.bincount(votes, minlength=2)
        preds.append(np.argmax(counts))
    return np.array(preds)

def accuracy(y_true, y_pred):
    return float(np.mean(y_true == y_pred))

## Bootstrap transfer helper

In [ ]:
def run_transfer_once(train_structured, train_poisson, test_structured, test_poisson):
    m_train = min(len(train_structured), len(train_poisson))
    m_test = min(len(test_structured), len(test_poisson))

    X_train = np.vstack([train_structured[:m_train], train_poisson[:m_train]])
    y_train = np.array([0] * m_train + [1] * m_train)

    X_test = np.vstack([test_structured[:m_test], test_poisson[:m_test]])
    y_test = np.array([0] * m_test + [1] * m_test)

    X_train_s, X_test_s = standardize_train_test(X_train, X_test)

    centroids = nearest_centroid_fit(X_train_s, y_train)
    y_pred_centroid = nearest_centroid_predict(X_test_s, centroids)
    y_pred_knn = knn_predict(X_train_s, y_train, X_test_s, k=5)

    return {
        "centroid_accuracy": accuracy(y_test, y_pred_centroid),
        "knn_accuracy": accuracy(y_test, y_pred_knn),
    }

def bootstrap_rows(X, rng):
    idx = rng.integers(0, len(X), size=len(X))
    return X[idx]

def bootstrap_transfer(train_structured, train_poisson, test_structured, test_poisson, n_boot=100, seed=9423):
    boot_rng = np.random.default_rng(seed)

    centroid_vals = []
    knn_vals = []

    for _ in range(n_boot):
        ts = bootstrap_rows(train_structured, boot_rng)
        tp = bootstrap_rows(train_poisson, boot_rng)
        vs = bootstrap_rows(test_structured, boot_rng)
        vp = bootstrap_rows(test_poisson, boot_rng)

        out = run_transfer_once(ts, tp, vs, vp)
        centroid_vals.append(out["centroid_accuracy"])
        knn_vals.append(out["knn_accuracy"])

    centroid_vals = np.array(centroid_vals)
    knn_vals = np.array(knn_vals)

    return {
        "centroid_mean": float(centroid_vals.mean()),
        "centroid_std": float(centroid_vals.std()),
        "centroid_q025": float(np.quantile(centroid_vals, 0.025)),
        "centroid_q975": float(np.quantile(centroid_vals, 0.975)),
        "knn_mean": float(knn_vals.mean()),
        "knn_std": float(knn_vals.std()),
        "knn_q025": float(np.quantile(knn_vals, 0.025)),
        "knn_q975": float(np.quantile(knn_vals, 0.975)),
    }

## Height blocks

In [ ]:
block_size = 300
block_starts = [0, 300, 600, 900, 1200]
block_labels = [f"{s+1}-{s+block_size}" for s in block_starts]

zeta_blocks = {}
poisson_blocks = {}

for s, label in zip(block_starts, block_labels):
    zeta_block = zeta_gaps_full[s:s + block_size]
    poisson_block = poisson_gaps_full[s:s + block_size]
    _, zeta_X = build_feature_matrix(zeta_block, k=5)
    _, poisson_X = build_feature_matrix(poisson_block, k=5)
    zeta_blocks[label] = zeta_X
    poisson_blocks[label] = poisson_X

_, gue_X_full = build_feature_matrix(gue_gaps_full[:1400], k=5)

block_labels

## Bootstrap the forward transfer directions

Train block:
- 1–300

Test blocks:
- 301–600
- 601–900
- 901–1200
- 1201–1500

In [ ]:
train_label = block_labels[0]
test_labels = block_labels[1:]

gue_to_zeta = []
zeta_to_gue = []
zeta_to_zeta = []
gue_to_gue = []

for i, test_label in enumerate(test_labels):
    train_poisson = poisson_blocks[train_label]
    test_poisson = poisson_blocks[test_label]

    zeta_train = zeta_blocks[train_label]
    zeta_test = zeta_blocks[test_label]

    m_train_gue = min(len(gue_X_full), len(train_poisson), len(zeta_train))
    m_test_gue = min(len(gue_X_full), len(test_poisson), len(zeta_test))

    gue_train = gue_X_full[:m_train_gue]
    gue_test = gue_X_full[-m_test_gue:]

    seed_base = 9423 + 100*i

    gue_to_zeta.append({
        "test_block": test_label,
        "summary": bootstrap_transfer(gue_train, train_poisson, zeta_test, test_poisson, n_boot=100, seed=seed_base + 1),
    })

    zeta_to_gue.append({
        "test_block": test_label,
        "summary": bootstrap_transfer(zeta_train, train_poisson, gue_test, test_poisson, n_boot=100, seed=seed_base + 2),
    })

    zeta_to_zeta.append({
        "test_block": test_label,
        "summary": bootstrap_transfer(zeta_train, train_poisson, zeta_test, test_poisson, n_boot=100, seed=seed_base + 3),
    })

    gue_to_gue.append({
        "test_block": test_label,
        "summary": bootstrap_transfer(gue_train, train_poisson, gue_test, test_poisson, n_boot=100, seed=seed_base + 4),
    })

gue_to_zeta[:1], zeta_to_gue[:1]

## k-NN transfer accuracies with bootstrap intervals

In [ ]:
x = np.arange(len(test_labels))

def means_qs(records, prefix="knn"):
    means = np.array([r["summary"][f"{prefix}_mean"] for r in records])
    lows = np.array([r["summary"][f"{prefix}_mean"] - r["summary"][f"{prefix}_q025"] for r in records])
    highs = np.array([r["summary"][f"{prefix}_q975"] - r["summary"][f"{prefix}_mean"] for r in records])
    return means, lows, highs

g2z_mean, g2z_low, g2z_high = means_qs(gue_to_zeta, "knn")
z2g_mean, z2g_low, z2g_high = means_qs(zeta_to_gue, "knn")
z2z_mean, z2z_low, z2z_high = means_qs(zeta_to_zeta, "knn")
g2g_mean, g2g_low, g2g_high = means_qs(gue_to_gue, "knn")

plt.figure(figsize=(11.2, 5.0))
plt.errorbar(x, z2z_mean, yerr=np.vstack([z2z_low, z2z_high]), marker="o", capsize=4, label="zeta→zeta baseline")
plt.errorbar(x, g2g_mean, yerr=np.vstack([g2g_low, g2g_high]), marker="o", capsize=4, label="GUE→GUE baseline")
plt.errorbar(x, g2z_mean, yerr=np.vstack([g2z_low, g2z_high]), marker="o", capsize=4, label="GUE→zeta")
plt.errorbar(x, z2g_mean, yerr=np.vstack([z2g_low, z2g_high]), marker="o", capsize=4, label="zeta→GUE")
plt.xticks(x, test_labels, rotation=25)
plt.ylim(0.4, 1.0)
plt.ylabel("k-NN accuracy")
plt.title("Cross-ensemble transfer with bootstrap intervals")
plt.legend()
plt.tight_layout()
plt.show()

## Nearest-centroid transfer accuracies with bootstrap intervals

In [ ]:
g2z_mean_c, g2z_low_c, g2z_high_c = means_qs(gue_to_zeta, "centroid")
z2g_mean_c, z2g_low_c, z2g_high_c = means_qs(zeta_to_gue, "centroid")
z2z_mean_c, z2z_low_c, z2z_high_c = means_qs(zeta_to_zeta, "centroid")
g2g_mean_c, g2g_low_c, g2g_high_c = means_qs(gue_to_gue, "centroid")

plt.figure(figsize=(11.2, 5.0))
plt.errorbar(x, z2z_mean_c, yerr=np.vstack([z2z_low_c, z2z_high_c]), marker="o", capsize=4, label="zeta→zeta baseline")
plt.errorbar(x, g2g_mean_c, yerr=np.vstack([g2g_low_c, g2g_high_c]), marker="o", capsize=4, label="GUE→GUE baseline")
plt.errorbar(x, g2z_mean_c, yerr=np.vstack([g2z_low_c, g2z_high_c]), marker="o", capsize=4, label="GUE→zeta")
plt.errorbar(x, z2g_mean_c, yerr=np.vstack([z2g_low_c, z2g_high_c]), marker="o", capsize=4, label="zeta→GUE")
plt.xticks(x, test_labels, rotation=25)
plt.ylim(0.4, 1.0)
plt.ylabel("nearest-centroid accuracy")
plt.title("Cross-ensemble transfer with bootstrap intervals")
plt.legend()
plt.tight_layout()
plt.show()

## Gap-to-baseline summary

These gaps measure how far cross-ensemble transfer sits below same-ensemble transfer.
Smaller gaps mean the transferred boundary is almost as good as the native one.

In [ ]:
gap_summary = []

for i, test_label in enumerate(test_labels):
    gap_summary.append({
        "test_block": test_label,
        "knn_gap_g2z_vs_z2z": float(z2z_mean[i] - g2z_mean[i]),
        "knn_gap_z2g_vs_g2g": float(g2g_mean[i] - z2g_mean[i]),
        "centroid_gap_g2z_vs_z2z": float(z2z_mean_c[i] - g2z_mean_c[i]),
        "centroid_gap_z2g_vs_g2g": float(g2g_mean_c[i] - z2g_mean_c[i]),
    })

gap_summary

## Compact summary table

In [ ]:
summary = {
    "gue_to_zeta": gue_to_zeta,
    "zeta_to_gue": zeta_to_gue,
    "zeta_to_zeta": zeta_to_zeta,
    "gue_to_gue": gue_to_gue,
    "gap_summary": gap_summary,
}
summary

## Notes

- If the bootstrap intervals for GUE→zeta stay close to the zeta→zeta baseline, that supports the idea that a boundary learned from GUE is genuinely informative for zeta.
- If zeta→GUE behaves similarly, the transfer is reciprocal rather than one-sided.
- This notebook focuses on one compact feature set and one training block. Later notebooks could vary those choices.

## Next directions

A natural next notebook could do one of these:

1. test several alternative minimal feature sets under the same bootstrap transfer protocol  
2. train on a middle block and test on both lower and higher blocks  
3. extend the bootstrap transfer idea to conditional windows  
4. compare linear vs nonlinear separators explicitly